# HTML Video

Browse bundled HTML video templates, inspect a template, and render configured WebM frames or media ports through the app MCP tools.

In [ ]:
// TODO(U7): replace with `import { callTool } from "jsr:@spur/app"` once published to JSR.
// Deno Jupyter supports file:// imports; the path is resolved relative to the notebook's
// working directory (the directory containing app.ipynb).
const { callTool } = await import(
  `file://${Deno.cwd()}/../../sdk/typescript/src/call_tool.ts`
);
// Expose callTool on globalThis so later cells in the same kernel session can use it
// without re-importing (Deno Jupyter cells share globalThis across executions).
(globalThis as any).callTool = callTool;

const searchResult = await callTool("html_video_search_templates", {
  intent: "product launch motion typography dashboard intro",
  top: 8
});
const items = Array.isArray(searchResult.items) ? searchResult.items : [];
const selectedId = items[0]?.id ?? "basic";
const { tableFromArrays } = await import("npm:apache-arrow@21.1.0");
const searchTable = tableFromArrays({
  id: items.map((i: any) => i.id ?? ""),
  title: items.map((i: any) => i.title ?? ""),
  intent: items.map((i: any) => i.intent ?? ""),
  summary: items.map((i: any) => i.summary ?? ""),
  score: items.map((i: any) => Number(i.score ?? 0)),
});
await spur.put("template_search", searchTable);

// TODO(U7): replace with `await ports.read(port)` from "jsr:@spur/app" once published.
// ports.ts imports @std/path which requires the deno.json import map — not available in
// the kernel context, so file:// import of ports.ts fails (confirmed by deno eval probe).
function readPortPayload(port: string): number[] | null {
  const root = Deno.env.get("SPUR_NOTEBOOK_PORT_ROOT");
  if (!root) return null;
  const manifest = JSON.parse(Deno.readTextFileSync(`${root}/ports/manifest.json`));
  const entryPath: string | undefined = manifest.ports?.[port]?.path;
  if (!entryPath) return null;
  // Basename-join under the ports directory — never use the raw manifest path
  // verbatim, which may be an absolute path that resolves against CWD instead.
  const basename = entryPath.includes("/") ? entryPath.split("/").pop()! : entryPath;
  return Array.from(Deno.readFileSync(`${root}/ports/${basename}`));
}

const selectionPayloads: Record<string, number[]> = {};
for (const item of items) {
  if (!item?.id) continue;
  await spur.put("template_selection", [{ id: item.id }]);
  const payload = readPortPayload("template_selection");
  if (payload) selectionPayloads[item.id] = payload;
}
await spur.put("template_selection", [{ id: selectedId }]);

const { widget } = await spur.anywidget();
widget({
  state: { items, selectedId, selectionPayloads },
  render({ model, el, experimental }: any) {
    const items = model.get("items") ?? [];
    const selectedId = model.get("selectedId");
    el.innerHTML = `
      <style>
        .hv-search{padding:24px;font:14px system-ui,sans-serif;color:#172033;background:#fbfcfe}
        .hv-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin-top:16px}
        .hv-card{border:1px solid #d9e2ec;background:white;border-radius:8px;padding:14px;box-shadow:0 1px 2px rgba(15,23,42,.04);cursor:pointer;text-align:left}
        .hv-card[data-selected="true"]{border-color:#2563eb;box-shadow:0 0 0 2px rgba(37,99,235,.16)}
        .hv-title{font-weight:700;margin-bottom:6px}.hv-tags{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}.hv-tag{font-size:11px;border:1px solid #dbe3ef;border-radius:999px;padding:2px 7px;background:#f8fafc}
      </style>
      <section class="hv-search">
        <h2>Template Browser</h2>
        <div class="hv-grid">
          ${items.map((item: any) => `<article class="hv-card" data-template-id="${item.id}" data-selected="${item.id === selectedId}"><div class="hv-title">${item.title ?? item.id}</div><div>${item.summary ?? item.intent ?? ""}</div><div class="hv-tags">${(item.tags ?? []).map((tag: any) => `<span class="hv-tag">${tag}</span>`).join("")}</div></article>`).join("")}
        </div>
      </section>`;
    el.querySelectorAll(".hv-card").forEach((card: any) => {
      card.addEventListener("click", async () => {
        const id = card.getAttribute("data-template-id");
        if (!id) return;
        model.set("selectedId", id);
        el.querySelectorAll(".hv-card").forEach((candidate: any) => {
          candidate.dataset.selected = String(candidate.getAttribute("data-template-id") === id);
        });
        const payload = model.get("selectionPayloads")?.[id];
        if (payload && experimental?.invoke) {
          try { await experimental.invoke("source.push", { port: "template_selection", payload }); } catch (_) {}
        }
      });
    });
  }
})

In [ ]:
// callTool is set on globalThis by the SDK-client cell (cell 1).
// Guard: fail fast with a clear message if that cell has not been run yet.
if (!(globalThis as any).callTool) throw new Error("Run the SDK-client cell (cell 1) before this cell.");
const callTool = (globalThis as any).callTool as typeof import("../../sdk/typescript/src/call_tool.ts").callTool;

function normalizeArrowValue(value: any): any {
  if (value === null || value === undefined) return value;
  if (typeof value.toJSON === "function") return value.toJSON();
  if (Array.isArray(value)) return value.map(normalizeArrowValue);
  if (typeof value === "object") {
    const output: Record<string, unknown> = {};
    for (const [key, item] of Object.entries(value)) output[key] = normalizeArrowValue(item);
    return output;
  }
  return value;
}

function tableRows(table: any): any[] {
  if (!table) return [];
  return table.toArray().map((row: any) => normalizeArrowValue(row));
}

function firstPortRow(port: string): any | null {
  try {
    return tableRows(spur.get(port))[0] ?? null;
  } catch (_) {
    return null;
  }
}

const search = firstPortRow("template_search") ?? { items: [] };
const items = Array.isArray(search.items) ? search.items : [];
const selection = firstPortRow("template_selection");
const templateId = selection?.id ?? items[0]?.id ?? "basic";
const template = await callTool("html_video_get_template", { id: templateId });
const { tableFromArrays: _tfa2 } = await import("npm:apache-arrow@21.1.0");
const templateTable = _tfa2({
  id: [template.metadata?.id ?? template.id ?? templateId],
  title: [template.metadata?.title ?? template.id ?? templateId],
  summary: [template.metadata?.summary ?? ""],
  html: [template.html ?? ""],
  skill_md: [template.skill_md ?? ""],
});
await spur.put("template_data", templateTable);

const { widget } = await spur.anywidget();
widget({
  state: {
    title: template.metadata?.title ?? template.id ?? templateId,
    id: template.metadata?.id ?? template.id ?? templateId,
    summary: template.metadata?.summary ?? "",
    html: template.html ?? "",
    skill: template.skill_md ?? ""
  },
  render({ model, el }: any) {
    const html = model.get("html") ?? "";
    el.innerHTML = `
      <style>
        .hv-preview{display:grid;grid-template-columns:minmax(280px,1fr) minmax(280px,1.2fr);gap:18px;padding:24px;font:14px system-ui,sans-serif;color:#172033;background:white}
        .hv-pane{min-height:220px;border:1px solid #d9e2ec;border-radius:8px;background:#fbfcfe;overflow:hidden}.hv-meta{padding:18px}.hv-meta h2{margin:0 0 8px}.hv-code{margin-top:14px;max-height:180px;overflow:auto;border:1px solid #e2e8f0;background:#0f172a;color:#e2e8f0;border-radius:6px;padding:12px;font:12px ui-monospace,monospace;white-space:pre-wrap}.hv-frame{width:100%;height:100%;min-height:360px;border:0;background:white}
      </style>
      <section class="hv-preview">
        <div class="hv-pane hv-meta"><h2>${model.get("title")}</h2><div>${model.get("summary")}</div><pre class="hv-code">${html.replace(/[<&]/g, (c: string) => c === "<" ? "&lt;" : "&amp;")}</pre></div>
        <div class="hv-pane"><iframe class="hv-frame" sandbox="allow-scripts" srcdoc="${html.replace(/&/g, "&amp;").replace(/"/g, "&quot;")}"></iframe></div>
      </section>`;
  }
})

In [ ]:
// callTool is set on globalThis by the SDK-client cell (cell 1).
// Guard: fail fast with a clear message if that cell has not been run yet.
if (!(globalThis as any).callTool) throw new Error("Run the SDK-client cell (cell 1) before this cell.");
const callTool = (globalThis as any).callTool as typeof import("../../sdk/typescript/src/call_tool.ts").callTool;

function normalizeArrowValue(value: any): any {
  if (value === null || value === undefined) return value;
  if (typeof value.toJSON === "function") return value.toJSON();
  if (Array.isArray(value)) return value.map(normalizeArrowValue);
  if (typeof value === "object") {
    const output: Record<string, unknown> = {};
    for (const [key, item] of Object.entries(value)) output[key] = normalizeArrowValue(item);
    return output;
  }
  return value;
}

function firstPortRow(port: string): any | null {
  try {
    return spur.get(port).toArray().map((row: any) => normalizeArrowValue(row))[0] ?? null;
  } catch (_) {
    return null;
  }
}

const template = firstPortRow("template_data");
const templateId = template?.metadata?.id ?? template?.id ?? "basic";
const defaultOutput = `${Deno.cwd()}/html-video-render.mp4`;
const request = null;
let renderResult = null;
let renderError = null;
if (request) {
  try {
    renderResult = await callTool("html_video_render", request);
  } catch (error) {
    renderError = error instanceof Error ? error.message : String(error);
  }
}

const { widget } = await spur.anywidget();
widget({
  state: { request, renderResult, renderError, defaultOutput, templateId },
  render({ model, el }: any) {
    const request = model.get("request");
    const result = model.get("renderResult");
    const error = model.get("renderError");
    el.innerHTML = `
      <style>
        .hv-render{padding:24px;font:14px system-ui,sans-serif;color:#172033;background:#f8fafc}.hv-panel{max-width:760px;border:1px solid #d9e2ec;border-radius:8px;background:white;padding:18px}.hv-row{display:grid;grid-template-columns:150px 1fr;gap:10px;align-items:center;margin-top:10px}.hv-input{border:1px solid #cbd5e1;border-radius:6px;padding:8px;background:#f8fafc;font:13px ui-monospace,monospace}.hv-status{margin-top:16px;border-radius:6px;padding:12px;background:#eef6ff;border:1px solid #bfdbfe}.hv-error{background:#fff1f2;border-color:#fecdd3}.hv-json{white-space:pre-wrap;font:12px ui-monospace,monospace}</style>
      <section class="hv-render"><div class="hv-panel"><h2>Render Controls</h2><div class="hv-row"><label>Template</label><div class="hv-input">${model.get("templateId")}</div></div><div class="hv-row"><label>Output path</label><div class="hv-input">${request?.output_path ?? model.get("defaultOutput")}</div></div><div class="hv-row"><label>Resolution</label><div class="hv-input">${request?.resolution ?? "1280x720"}</div></div><div class="hv-row"><label>FPS</label><div class="hv-input">${request?.fps ?? 30}</div></div>${result ? `<div class="hv-status"><strong>Rendered</strong><pre class="hv-json">${JSON.stringify(result, null, 2)}</pre></div>` : error ? `<div class="hv-status hv-error"><strong>Render error</strong><pre class="hv-json">${error}</pre></div>` : `<div class="hv-status"><strong>Ready</strong><div>Render request is waiting for webm_frames or port_names.</div></div>`}</div></section>`;
  }
})

In [ ]:
const adHtml = `<!doctype html>
<html>
<head>
  <meta charset="UTF-8" />
  <style>
    html, body {
      margin: 0;
      padding: 0;
      width: 100%;
      height: 100%;
      background: #040711;
    }

    body {
      overflow: hidden;
      display: flex;
      align-items: center;
      justify-content: center;
      color: #e2e8f0;
      font-family: "Inter", "Segoe UI", sans-serif;
    }

    canvas {
      width: 1280px;
      height: 720px;
      display: block;
      margin: 0 auto;
      border: 1px solid #1d2a42;
      background: #081325;
    }
  </style>
</head>
<body>
  <canvas data-capture="true" data-capture-cell-id="spur-ad-capture" data-capture-fps="30" data-capture-duration-sec="60" width="1280" height="720"></canvas>

  <script>
    (() => {
      const canvas = document.querySelector('canvas[data-capture="true"]');
      if (!canvas) return;
      const ctx = canvas.getContext('2d');
      if (!ctx) return;

      const W = canvas.width;
      const H = canvas.height;
      const DURATION = 60000;
      const startT = performance.now();

      const clamp = (value, min, max) => Math.max(min, Math.min(max, value));
      const easeIn = (t) => t * t;
      const easeOut = (t) => 1 - Math.pow(1 - t, 3);
      const lerp = (a, b, t) => a + (b - a) * t;

      const sceneDefs = [
        [0, 6000, "problem"],
        [6000, 12000, "pain"],
        [12000, 18000, "lock"],
        [18000, 25000, "ledger"],
        [25000, 32000, "chaos"],
        [32000, 39000, "turn"],
        [39000, 46000, "workflow"],
        [46000, 53000, "sync"],
        [53000, 60000, "cta"],
      ];

      function drawBackground(progress) {
        const g = ctx.createLinearGradient(0, 0, 0, H);
        g.addColorStop(0, '#050812');
        g.addColorStop(0.5, '#07162b');
        g.addColorStop(1, '#02070d');
        ctx.fillStyle = g;
        ctx.fillRect(0, 0, W, H);

        ctx.globalAlpha = 0.35;
        for (let i = 0; i < 5; i++) {
          const x = (W / 6) * (i + 1) + Math.sin(progress * 0.00045 + i) * 14;
          const y = 90 + Math.cos(progress * 0.0006 + i * 0.5) * 14;
          const radius = 140 + i * 11;
          const g2 = ctx.createRadialGradient(x, y, 0, x, y, radius);
          g2.addColorStop(0, i % 2 ? 'rgba(103,232,249,0.20)' : 'rgba(250,204,21,0.16)');
          g2.addColorStop(1, 'rgba(3,6,12,0)');
          ctx.fillStyle = g2;
          ctx.beginPath();
          ctx.arc(x, y, radius, 0, Math.PI * 2);
          ctx.fill();
        }
        ctx.globalAlpha = 1;
      }

      function title(text, size, y, color, alpha = 1) {
        ctx.save();
        ctx.globalAlpha = alpha;
        ctx.textAlign = 'center';
        ctx.textBaseline = 'middle';
        ctx.fillStyle = color;
        ctx.font = `700 ${size}px 'Trebuchet MS', 'Segoe UI', sans-serif`;
        ctx.fillText(text, W * 0.5, y);
        ctx.restore();
      }

      function bodyText(text, size, y, color, alpha = 1) {
        ctx.save();
        ctx.globalAlpha = alpha;
        ctx.textAlign = 'center';
        ctx.textBaseline = 'middle';
        ctx.fillStyle = color;
        ctx.font = `${size}px 'Trebuchet MS', 'Segoe UI', sans-serif`;
        ctx.fillText(text, W * 0.5, y);
        ctx.restore();
      }

      function chip(x, y, w, h, label, color, alpha = 1) {
        ctx.save();
        ctx.globalAlpha = alpha;
        ctx.fillStyle = 'rgba(15, 23, 42, 0.70)';
        ctx.strokeStyle = color;
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(x + 14, y);
        ctx.arcTo(x + w, y, x + w, y + h, 14);
        ctx.arcTo(x + w, y + h, x, y + h, 14);
        ctx.arcTo(x, y + h, x, y, 14);
        ctx.arcTo(x, y, x + w, y, 14);
        ctx.closePath();
        ctx.fill();
        ctx.stroke();
        ctx.fillStyle = color;
        ctx.font = '700 21px Menlo, Monaco, monospace';
        ctx.textAlign = 'center';
        ctx.textBaseline = 'middle';
        ctx.fillText(label, x + w * 0.5, y + h * 0.5);
        ctx.restore();
      }

      function sceneProblem(t, p) {
        drawBackground(t * 0.001);
        title('5 AGENTS, MANY WINDOWS', 62, 120, '#67e8f9', clamp(easeIn(p), 0, 1));
        bodyText('Every context switch breaks your momentum.', 34, H * 0.43, '#e2e8f0', clamp(p, 0, 1));
        bodyText('One terminal is not an operations dashboard.', 30, H * 0.57, '#cbd5e1', clamp(p - 0.1, 0, 1));
      }

      function scenePain(t, p) {
        drawBackground(t * 0.001);
        title('PAIN POINTS', 56, 118, '#f0abfc', clamp(easeOut(p), 0, 1));
        const items = [
          'README drift and hidden state in every session',
          'Rate-limit pauses between retries',
          'Cost surprises after each run',
        ];
        items.forEach((item, idx) => {
          chip(150, 210 + idx * 145, 980, 88, item, '#67e8f9', clamp(p - idx * 0.15, 0, 1));
        });
        bodyText('You need orchestration, not another tab.', 28, H - 80, '#7dd3fc', clamp(p, 0, 1));
      }

      function sceneLock(t, p) {
        drawBackground(t * 0.001);
        title('RATE LIMIT AMBUSH', 58, 120, '#f59e0b', clamp(easeOut(p), 0, 1));
        const barW = W - 320;
        const progress = clamp(p, 0, 1);
        ctx.fillStyle = '#0f172a';
        ctx.fillRect(160, H * 0.45, barW, 56);
        ctx.fillStyle = '#ef4444';
        const width = lerp(0, barW, progress);
        ctx.fillRect(160, H * 0.45, width, 56);
        bodyText('Requests stall. Workers starve. CI waits.', 30, H * 0.58, '#e2e8f0', clamp(p, 0, 1));
      }

      function sceneLedger(t, p) {
        drawBackground(t * 0.001);
        title('ONE COST LEDGER', 56, 110, '#34d399', clamp(easeOut(p), 0, 1));
        const rows = [
          ['CLAUDE', 0.92, '#67e8f9'],
          ['CODEX', 0.64, '#f0abfc'],
          ['GEMINI', 0.41, '#f472b6'],
          ['ANOTHER', 0.27, '#7dd3fc'],
        ];
        rows.forEach((row, i) => {
          const y = 230 + i * 95;
          const w = (W - 360) * row[1];
          bodyText(row[0], 29, y, '#e2e8f0', clamp(p - i * 0.12, 0, 1));
          ctx.fillStyle = 'rgba(255,255,255,0.08)';
          ctx.fillRect(320, y + 16, W - 340, 28);
          ctx.fillStyle = row[2];
          ctx.fillRect(320, y + 16, w * clamp(easeIn(p), 0, 1), 28);
        });
        const total = Math.round(lerp(0, 1284, clamp(p * 1.1, 0, 1)));
        bodyText(`TOTAL COST (ONE MONTH): $${total.toLocaleString()}`, 36, H - 95, '#a5f3fc', clamp(p, 0, 1));
      }

      function sceneChaos(t, p) {
        drawBackground(t * 0.001);
        title('MULTI-AGENT CHAOS', 56, 114, '#c084fc', clamp(easeOut(p), 0, 1));
        const points = 26;
        for (let i = 0; i < points; i++) {
          const phase = i * 0.9 + t * 0.00095;
          const x = (W * 0.45) + Math.cos(phase) * (300 + (i % 4) * 30);
          const y = H * 0.5 + Math.sin(phase * 1.15) * (90 + (i % 5) * 14);
          const r = 5 + (i % 3) * 1.8;
          ctx.fillStyle = `hsla(${180 + (i * 11) % 120}, 80%, 72%, 0.8)`;
          ctx.beginPath();
          ctx.arc(x, y, r, 0, Math.PI * 2);
          ctx.fill();
        }
        bodyText('Which worker is alive? Which one is blocked?', 30, H - 120, '#e2e8f0', clamp(p, 0, 1));
      }

      function sceneTurn(t, p) {
        drawBackground(t * 0.001);
        const alpha = clamp(easeOut(p), 0, 1);
        title('TURN THE INCIDENT INTO FLOW', 56, 118, '#67e8f9', alpha);
        chip(270, 240, 240, 70, 'ISSUE', '#34d399', alpha);
        chip(520, 240, 260, 70, 'BRAIN', '#f472b6', alpha);
        chip(790, 240, 300, 70, 'SPUR', '#93c5fd', alpha);
        bodyText('Plans, reviews, resumable execution, and governance in one place.', 28, H - 98, '#dbeafe', alpha);
      }

      function sceneWorkflow(t, p) {
        drawBackground(t * 0.001);
        title('WORKFLOW THAT DOES NOT COLLAPSE', 54, 112, '#a7f3d0', clamp(easeOut(p), 0, 1));
        const cards = [
          'ISSUE → BRAIN',
          'PLAN → BATCH',
          'WORKER → ARTIFACTS',
          'PR → REVIEW',
        ];
        cards.forEach((label, i) => {
          const x = 140 + i * 250;
          chip(x, 260 + Math.sin((t + i * 700) * 0.001) * 20, 220, 90, label, '#67e8f9', clamp(p - i * 0.1, 0, 1));
        });
      }

      function sceneSync(t, p) {
        drawBackground(t * 0.001);
        title('SYNC, RESUME, SWAP', 56, 110, '#67e8f9', clamp(easeOut(p), 0, 1));
        const messages = [
          'BRAIN-SWAP: continue from any model instantly.',
          'SESSION RESUME: pause the laptop, pick up where you left off.',
          'LOCAL-FIRST: state and plans stay durable on disk.',
        ];
        messages.forEach((msg, i) => {
          bodyText(msg, 28, 260 + i * 90, '#bfdbfe', clamp(p - i * 0.11, 0, 1));
        });
      }

      function sceneCta(t, p) {
        drawBackground(t * 0.001);
        title('SPUR', 88, H * 0.40, '#67e8f9', clamp(easeOut(p), 0, 1));
        title('The control tower for your CLI coding agents', 30, H * 0.56, '#e2e8f0', clamp(p, 0, 1));
        bodyText('getspur.dev', 54, H * 0.70, '#8b5cf6', clamp(Math.max(0, p - 0.2), 0, 1));
        bodyText('curl -sSL https://getspur.dev/install.sh | sh', 30, H * 0.80, '#d4d4d8', clamp(Math.max(0, p - 0.2), 0, 1));
      }

      function renderScene(progress) {
        const active = sceneDefs.find(([start, end]) => progress >= start && progress < end);
        if (!active) {
          sceneCta(progress, 1);
          return;
        }
        const [start, end, label] = active;
        const localP = clamp((progress - start) / (end - start), 0, 1);
        if (label === 'problem') sceneProblem(progress, localP);
        if (label === 'pain') scenePain(progress, localP);
        if (label === 'lock') sceneLock(progress, localP);
        if (label === 'ledger') sceneLedger(progress, localP);
        if (label === 'chaos') sceneChaos(progress, localP);
        if (label === 'turn') sceneTurn(progress, localP);
        if (label === 'workflow') sceneWorkflow(progress, localP);
        if (label === 'sync') sceneSync(progress, localP);
        if (label === 'cta') sceneCta(progress, localP);
      }

      const frame = (now) => {
        const elapsed = Math.min(now - startT, DURATION);
        renderScene(elapsed);

        ctx.fillStyle = '#9ca3af';
        ctx.fillRect(0, H - 8, W, 8);
        const done = elapsed / DURATION;
        ctx.fillStyle = '#67e8f9';
        ctx.fillRect(0, H - 8, W * done, 8);
        ctx.fillStyle = 'rgba(226,232,240,0.9)';
        ctx.textAlign = 'left';
        ctx.textBaseline = 'alphabetic';
        ctx.font = '19px Monaco, Menlo, monospace';
        ctx.fillText(`0:${String(Math.floor(elapsed / 1000)).padStart(2, '0')} / 1:00`, 24, H - 20);

        if (elapsed < DURATION) {
          requestAnimationFrame(frame);
        }
      };

      requestAnimationFrame(frame);
    })();
  </script>
</body>
</html>`;

({
  [Symbol.for("Jupyter.display")]: () => {
    return {
      "text/html": adHtml,
    };
  },
})


In [ ]:
// callTool is set on globalThis by the SDK-client cell (cell 1).
// Guard: fail fast with a clear message if that cell has not been run yet.
if (!(globalThis as any).callTool) throw new Error("Run the SDK-client cell (cell 1) before this cell.");
const callTool = (globalThis as any).callTool as typeof import("../../sdk/typescript/src/call_tool.ts").callTool;

const outputPath = `${Deno.cwd()}/spur-ad.mp4`;
const renderRequest = {
  port_names: ["spur-ad-capture"],
  output_path: outputPath,
  resolution: "1280x720",
  fps: 30,
  frame_duration: 60,
};

let renderResult = null;
let renderError = null;
try {
  renderResult = await callTool("html_video_render", renderRequest);
  const resolved = renderResult?.output_path ?? outputPath;
  await Deno.writeTextFile("spur-ad-output-path.txt", resolved).catch(() => {});
} catch (error) {
  const message = error instanceof Error ? error.message : String(error);
  if (message.includes("could not read media port") || message.includes("not found")) {
    renderError = "capture not ready. Run the capture cell and allow 60 seconds to complete before rendering.";
  } else {
    renderError = message;
  }
}

const status = renderError
  ? `<div style="font:14px/1.4 system-ui,sans-serif;color:#0f172a">⚠️ ${renderError}</div>`
  : `<div style="font:14px/1.4 system-ui,sans-serif;color:#0f172a"><strong>Render complete.</strong><div style="margin-top:8px">output_path: ${renderResult?.output_path ?? outputPath}</div></div>`;

({
  [Symbol.for("Jupyter.display")]: () => {
    return {
      "text/html": `<!doctype html><div style="padding:16px;background:#f8fafc;border:1px solid #dbeafe;border-radius:10px">${status}</div>`,
    };
  },
});

In [ ]:
let outputPath = `${Deno.cwd()}/spur-ad.mp4`;
try {
  const cached = Deno.readTextFileSync("spur-ad-output-path.txt").trim();
  if (cached.length > 0) outputPath = cached;
} catch (_) {
  // no-op: use default output path
}

function encodeBase64(bytes) {
  let binary = "";
  const chunk = 0x8000;
  for (let i = 0; i < bytes.length; i += chunk) {
    binary += String.fromCharCode(...bytes.subarray(i, i + chunk));
  }
  return btoa(binary);
}

let videoBase64 = null;
let embedError = null;
try {
  const data = await Deno.readFile(outputPath);
  videoBase64 = encodeBase64(data);
} catch (error) {
  const message = error instanceof Error ? error.message : String(error);
  if (message.includes("No such file") || message.includes("not found") || message.includes("No such")) {
    embedError = `render output not available yet at ${outputPath}. Run render cell first.`;
  } else {
    embedError = message;
  }
}

const output = videoBase64
  ? {
      [Symbol.for("Jupyter.display")]: () => ({
        "text/html": `<!doctype html><video controls playsinline style="max-width:100%" src="data:video/mp4;base64,${videoBase64}"></video>`,
      }),
    }
  : {
      [Symbol.for("Jupyter.display")]: () => ({
        "text/html": `<!doctype html><div style="padding:12px;border:1px solid #fecdd3;background:#fff1f2;border-radius:8px;color:#881337">⚠️ ${embedError}</div>`,
      }),
    };

output;
